In [1]:
!pip install --quiet scikit-learn imbalanced-learn matplotlib seaborn upsetplot tqdm



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ------------------------------------------------------------
#  Lung-cancer public-data bias study: visualisation bundle
#  -----------------------------------------------------------
#  Prereqs: rna2.csv, sample_sheet.tsv, clinical/clinical.tsv
#           built exactly as in your current notebook.
# ------------------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

# ML & stats
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import mutual_info_classif, SelectKBest

# Plots
import matplotlib.pyplot as plt
import seaborn as sns
from upsetplot import UpSet, from_memberships

# -------- 1. LOAD & MERGE -----------------------------------------------------

# RNA expression (z-scored + clipped)
rna = pd.read_csv("rna2.csv")                       # rows = samples, cols = genes + 'Sample Type'
print(f"RNA matrix shape  : {rna.shape}")

# Sample sheet – maps file name → sample type + Case ID
sample_sheet = pd.read_csv("sample_sheet.tsv", sep="\t")

# Clinical – demographics at case level
clinical = pd.read_csv(Path("clinical") / "clinical.tsv", sep="\t")
# Merge all: rna → sample_sheet on File Name; then → clinical on Case ID vs case_submitter_id
rna = (
    rna
      .merge(sample_sheet[["File Name", "Case ID"]], left_on="case_id", right_on="File Name")
      .merge(
          clinical[
              ["case_submitter_id", "gender", "race", "ethnicity", "age_at_index"]
          ],
          left_on="Case ID",
          right_on="case_submitter_id",
          how="left"
      )
      .drop(columns=["File Name", "case_submitter_id"])
)

# Keep only Primary Tumour (1) & Normal (0) label column already encoded
y = rna["Sample Type"]
X_full = rna.drop(columns=["case_id", "Sample Type"])   # features + demographics for now

# Demographic columns for fairness
demo_cols = ["gender", "race", "ethnicity"]

# -------- 2. CROSS-VALIDATION & DATA COLLECTION ------------------------------

seed = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
model_core = LogisticRegression(max_iter=1000, solver="liblinear")

# CONFIGS (edit if you like)
K_GENES = 100               # top-K MI genes per fold (instability analysis)
OVERFITTING_METRIC = "AUC"  # what to plot on y-axis
FAIRNESS_THRESH = 0.10      # bold if |metric - overall| > threshold

# Containers for plots / tables
train_scores, val_scores = [], []
feature_sets = []           # list of sets of selected gene names per fold
fairness_records = []       # one row per sample for subgroup metrics

for fold, (train_idx, val_idx) in enumerate(cv.split(X_full, y), start=1):
    X_train, X_val = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # ---- 2a. Feature selection on *training* only (mutual info) -------------
    mi = mutual_info_classif(X_train.drop(columns=demo_cols), y_train, random_state=seed)
    mi_series = pd.Series(mi, index=X_train.drop(columns=demo_cols).columns)
    top_k_genes = mi_series.sort_values(ascending=False).head(K_GENES).index.to_list()
    feature_sets.append(set(top_k_genes))

    # Reduce to genes only (demographics excluded from classifier on purpose)
    Xtr = X_train[top_k_genes]
    Xte = X_val[top_k_genes]

    # ---- 2b. Build pipeline with SMOTE + logistic regression ----------------
    pipe = Pipeline(steps=[("smote", SMOTE(random_state=seed)),
                          ("clf"  , model_core)])

    pipe.fit(Xtr, y_train)

    # ---- 2c. Training & validation AUC --------------------------------------
    ytr_prob = pipe.predict_proba(Xtr)[:, 1]
    yte_prob = pipe.predict_proba(Xte)[:, 1]

    train_scores.append(roc_auc_score(y_train, ytr_prob))
    val_scores  .append(roc_auc_score(y_val  , yte_prob))

    # ---- 2d. Store per-sample outputs for fairness table --------------------
    yte_pred = (yte_prob >= 0.5).astype(int)

    fold_df = pd.DataFrame({
        "fold" : fold,
        "y_true": y_val.values,
        "y_pred": yte_pred,
        "prob"  : yte_prob
    }).join(X_val[demo_cols].reset_index(drop=True))

    fairness_records.append(fold_df)

fairness_df = pd.concat(fairness_records, ignore_index=True)

# -------- 3. FIGURE 1 – Overfitting box/violin plot --------------------------

plt.figure(figsize=(6, 4))
plot_df = pd.DataFrame({
    "AUC": train_scores + val_scores,
    "Dataset": ["Train"]*len(train_scores) + ["Validation"]*len(val_scores)
})
sns.violinplot(x="Dataset", y="AUC", data=plot_df, inner="box", palette="Set2")
plt.title("5-fold CV: Train vs Validation AUC\n(Lung-cancer RNA, Logistic Regression)")
plt.ylim(0.0, 1.0)
plt.show()

# -------- 4. FIGURE 2 – Fairness metrics by subgroup ------------------------

def subgroup_table(df, group_col):
    """Return Accuracy, FPR, FNR by subgroup"""
    records = []
    for grp, gdf in df.groupby(group_col):
        tp = ((gdf.y_true == 1) & (gdf.y_pred == 1)).sum()
        tn = ((gdf.y_true == 0) & (gdf.y_pred == 0)).sum()
        fp = ((gdf.y_true == 0) & (gdf.y_pred == 1)).sum()
        fn = ((gdf.y_true == 1) & (gdf.y_pred == 0)).sum()

        acc = (tp + tn) / len(gdf)
        fpr = fp / (fp + tn) if (fp + tn) else np.nan
        fnr = fn / (fn + tp) if (fn + tp) else np.nan

        records.append((grp, len(gdf), acc, fpr, fnr))
    return pd.DataFrame(records,
                        columns=[group_col, "N", "Accuracy", "FPR", "FNR"]).sort_values("N", ascending=False)


overall = {
    "Accuracy": (fairness_df.y_true == fairness_df.y_pred).mean(),
    "FPR"     : ( (fairness_df.y_true==0) & (fairness_df.y_pred==1) ).sum()
                 / (fairness_df.y_true==0).sum(),
    "FNR"     : ( (fairness_df.y_true==1) & (fairness_df.y_pred==0) ).sum()
                 / (fairness_df.y_true==1).sum()
}

def bold_disparities(val, metric_name):
    delta = val - overall[metric_name]
    return "font-weight:bold" if abs(delta) > FAIRNESS_THRESH else ""

# Build combined table for gender & race (extend if needed)
tables = {}
for col in ["gender", "race"]:
    tbl = subgroup_table(fairness_df, col)
    tables[col] = tbl.style.format({"Accuracy":"{:.3f}", "FPR":"{:.3f}", "FNR":"{:.3f}"}) \
                           .applymap(lambda v: bold_disparities(v, "Accuracy"), subset=["Accuracy"]) \
                           .applymap(lambda v: bold_disparities(v, "FPR"),      subset=["FPR"]) \
                           .applymap(lambda v: bold_disparities(v, "FNR"),      subset=["FNR"])
    display(f"=== {col.upper()} ===")
    display(tables[col])

# -------- 5. FIGURE 3 – Feature-instability UpSet & Jaccard heat-map ---------

# --- 5a. UpSet plot ----------------------------------------------------------
memberships = []
for idx, gene_set in enumerate(feature_sets, start=1):
    memberships.extend([[f"Fold {idx}" if g in gene_set else None for g in gene_set]])

# Create membership list of tuples for UpSet
all_genes = sorted(set.union(*feature_sets))
membership_tuples = []
for g in all_genes:
    membership_tuples.append(tuple(f"Fold {i+1}" for i, s in enumerate(feature_sets) if g in s))

upset_data = from_memberships(membership_tuples)
upset = UpSet(upset_data, subset_size="count", show_counts=True)
plt.figure(figsize=(7, 4))
upset.plot()
plt.suptitle(f"UpSet plot – intersection of top-{K_GENES} MI genes across folds")
plt.show()

# --- 5b. Jaccard heat-map ----------------------------------------------------
def jaccard(a, b):
    return len(a & b) / len(a | b)

n_folds = len(feature_sets)
jmat = np.zeros((n_folds, n_folds))
for i in range(n_folds):
    for j in range(n_folds):
        jmat[i, j] = jaccard(feature_sets[i], feature_sets[j])

plt.figure(figsize=(5, 4))
sns.heatmap(jmat, annot=True, cmap="viridis",
            xticklabels=[f"F{i+1}" for i in range(n_folds)],
            yticklabels=[f"F{i+1}" for i in range(n_folds)])
plt.title(f"Pairwise Jaccard similarity of top-{K_GENES} genes")
plt.tight_layout()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'rna2.csv'